Preprocessing Patient Dataset: Cleaning, Feature Engineering, and Encoding for Machine Learning

In [1]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

# Load dataset
patients = pd.read_csv("mock_patient_dataset.csv")

# Fill missing multi-labels
multi_fields = ['allergies', 'deficiencies', 'diseases']
for col in multi_fields:
    patients[col] = patients[col].fillna("")

# BMI
patients["height_m"] = patients["height"] / 100
patients = patients[patients["height_m"] > 0]  # Avoid division by zero
patients["bmi"] = patients["weight"] / (patients["height_m"] ** 2)

def bmi_category(bmi):
    if bmi < 18.5:
        return "Underweight"
    elif 18.5 <= bmi < 24.9:
        return "Normal"
    elif 25 <= bmi < 29.9:
        return "Overweight"
    else:
        return "Obese"

patients["bmi_category"] = patients["bmi"].apply(bmi_category)

# Split multi-label fields
for col in multi_fields:
    patients[col] = patients[col].apply(lambda x: [item.strip() for item in x.split(";") if item.strip()])

# Multi-hot encode multi-label fields
mlb = MultiLabelBinarizer()
for col in multi_fields:
    bin_df = pd.DataFrame(mlb.fit_transform(patients[col]), columns=[f"{col}_{cls}" for cls in mlb.classes_])
    patients = pd.concat([patients, bin_df], axis=1)

# One-hot encode categorical fields
cat_cols = ['gender', 'diet_preference', 'activity_level', 'bmi_category']
patients = pd.get_dummies(patients, columns=cat_cols)

# Drop temp column
patients.drop(columns=["height_m"], inplace=True)

# Save
patients.to_pickle("processed_patients.pkl")


In [2]:
(patients.head())


,age,weight,height,allergies,deficiencies,diseases,bmi,allergies_Eggs,allergies_Gluten,allergies_Lactose (Dairy),...,diet_preference_Veg,diet_preference_Vegan,activity_level_Active,activity_level_Light,activity_level_Moderate,activity_level_Sedentary,bmi_category_Normal,bmi_category_Obese,bmi_category_Overweight,bmi_category_Underweight
0,20,116,157,"[Eggs, Gluten]",[Zinc],[],47.060733,1,1,0,...,False,False,False,False,True,False,False,True,False,False
1,61,93,152,[],[Calcium],[Obesity],40.252770,0,0,0,...,False,False,False,False,True,False,False,True,False,False
2,24,96,181,"[Lactose (Dairy), Gluten]",[],"[Hypertension, Diabetes]",29.303135,0,1,1,...,True,False,False,True,False,False,False,False,True,False
3,44,92,146,[Lactose (Dairy)],[Iron],"[Fatty Liver, Diabetes]",43.160068,0,0,1,...,False,True,False,False,False,True,False,True,False,False
4,56,88,169,[],[],"[Diabetes, Hypothyroidism]",30.811246,0,0,0,...,False,False,False,False,True,False,False,True,False,False


In [3]:
file_path = "mock_indian_food_dataset.csv"
bad_lines = []

with open(file_path, "rb") as f:
    for i, line in enumerate(f, 1):
        try:
            line.decode("utf-8")  # Try decoding each line
        except UnicodeDecodeError as e:
            print(f"Line {i} has a decoding error: {e}")
            bad_lines.append((i, line))

# Print the bad lines (decoded using ISO-8859-1 to inspect content)
print("\n--- Problematic Lines ---")
for i, raw_line in bad_lines:
    try:
        decoded_line = raw_line.decode("ISO-8859-1")  # safer fallback
    except Exception as e:
        decoded_line = f"<Cannot decode: {e}>"
    print(f"Line {i}: {decoded_line.strip()}")



--- Problematic Lines ---


Preprocessing Indian Food Dataset: Cleaning, Encoding, and Diet Tag Classification for ML

In [4]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

# Load dataset
foods = pd.read_csv("mock_indian_food_dataset.csv")

# Fill and split multi-label columns
for col in ['allergens', 'tags', 'diet_Type']:
    foods[col] = foods[col].fillna("").astype(str)
    # Use "," as separator for 'tags' and 'diet_Type' columns
    delimiter = ";" if col in ['allergens'] else ","
    foods[col] = foods[col].apply(lambda x: [i.strip().lower() for i in x.split(delimiter) if i.strip()])

# Clean numeric columns
nutrient_cols = ["calories", "protein", "fat", "carbs"]
foods.dropna(subset=nutrient_cols, how='all', inplace=True)
for col in nutrient_cols:
    if foods[col].isnull().any():
        foods[col].fillna(foods[col].median(), inplace=True)

# Limit high-calorie meals
foods = foods[foods['calories'] < 1500]

# Normalize meal_type
foods["meal_type"] = foods["meal_type"].str.lower().str.strip()

# Avoid overly aggressive deduplication
foods.drop_duplicates(subset=["name", "meal_type", "calories"], inplace=True)

# Reset index
foods.reset_index(drop=True, inplace=True)

# Multi-hot encode allergens, tags, and diet_Type
for col in ["allergens", "tags", "diet_Type"]:
    foods[col] = foods[col].apply(lambda x: x if isinstance(x, list) else [])
    
    mlb = MultiLabelBinarizer()
    bin_data = mlb.fit_transform(foods[col])
    bin_df = pd.DataFrame(bin_data, columns=[f"{col}_{cls}" for cls in mlb.classes_])
    foods = pd.concat([foods, bin_df], axis=1)

# Save processed version
foods.to_pickle("processed_foods.pkl")


In [5]:
foods.head()

,name,meal_type,calories,protein,carbs,fat,allergens,tags,diet_Type,allergens_egg,...,tags_protein,tags_vitamin a,tags_vitamin b12,tags_vitamin d,tags_weight loss,tags_zinc,diet_Type_non-veg,diet_Type_veg,diet_Type_vegan,diet_Type_vegan.veg
0,Til Ladoo,breakfast,99.2,6.8,64.6,25.6,[sesame],"[vitamin b12, pcos/pcod, vitamin d, diabetes, ...",[veg],0,...,1,0,1,1,0,0,0,1,0,0
1,Cabbage Poriyal,snack,109.4,25.1,39.3,12.0,[],"[diabetes, hypothyroidism]","[veg, vegan]",0,...,0,0,0,0,0,0,0,1,1,0
2,Kadhi Chawal,lunch,330.3,14.1,91.3,24.3,[],"[fatty liver, zinc]",[veg],0,...,0,0,0,0,0,1,0,1,0,0
3,Oats Idli,lunch,188.9,17.8,27.3,26.8,[],"[gastritis, vitamin b12, heart disease]","[veg, vegan, non-veg]",0,...,0,0,1,0,0,0,1,1,1,0
4,Paneer Tikka,breakfast,476.7,9.5,13.9,9.8,"[soy, sesame]","[pcos/pcod, zinc, iron, gastritis, calcium]",[veg],0,...,0,0,0,0,0,1,0,1,0,0


Checking for rows in both the datasets

In [6]:
print("Patients dataset rows:", patients.shape[0])
print("Foods dataset rows:", foods.shape[0])


Patients dataset rows: 1000
Foods dataset rows: 634


 Generating Patient-Food Training Data: Feature Merging and Suitability Labeling for ML Model

In [7]:
import pandas as pd

# Load preprocessed data
patients = pd.read_pickle("processed_patients.pkl")
foods = pd.read_pickle("processed_foods.pkl")

# Sample for performance
patient_samples = patients.sample(n=600, random_state=42).copy()
food_samples = foods.sample(n=600, random_state=42).copy()

rows = []

for _, p_row in patient_samples.iterrows():
    for _, f_row in food_samples.iterrows():
        patient_id = p_row.name
        food_id = f_row.name

        # === Safety: Ensure all list fields are truly lists ===
        patient_allergies = p_row.get('allergies', [])
        patient_diseases = p_row.get('diseases', [])
        patient_deficiencies = p_row.get('deficiencies', [])
        food_allergens = f_row.get('allergens', [])
        food_tags = f_row.get('tags', [])

        if not isinstance(patient_allergies, list): patient_allergies = []
        if not isinstance(patient_diseases, list): patient_diseases = []
        if not isinstance(patient_deficiencies, list): patient_deficiencies = []
        if not isinstance(food_allergens, list): food_allergens = []
        if not isinstance(food_tags, list): food_tags = []

        # === Allergen conflict ===
        allergy_conflict = bool(set(patient_allergies) & set(food_allergens))

        # === Diet preference check ===
        # === Diet preference check ===
        diet_check = True
        tag_set = set(t.lower() for t in food_tags)

        if p_row.get("diet_preference_Vegan", 0) == 1:
            diet_check = "vegan" in tag_set
        elif p_row.get("diet_preference_Veg", 0) == 1:
            diet_check = bool(tag_set & {"veg", "vegan"})


        # === Disease tag match ===
        disease_match = any(d.lower() in [t.lower() for t in food_tags] for d in patient_diseases)

        # === Deficiency tag match ===
        deficiency_match = any(d.lower() in [t.lower() for t in food_tags] for d in patient_deficiencies)

        has_conditions = bool(patient_diseases or patient_deficiencies)

        # === Suitability ===
        suitable = (
            not allergy_conflict and
            diet_check and
            (disease_match or deficiency_match or not has_conditions)
        )

        label = int(suitable)

        # === Build feature row ===
        row = {
            "patient_id": patient_id,
            "food_id": food_id,
            "age": p_row["age"],
            "weight": p_row["weight"],
            "height": p_row["height"],
            "bmi": p_row["bmi"],
            "meal_type": f_row["meal_type"],
            "food_calories": f_row["calories"],
            "food_protein": f_row["protein"],
            "food_fat": f_row["fat"],
            "food_carbs": f_row["carbs"],
            "label": label
        }

        # Add encoded patient features
        for col in patients.columns:
            if col.startswith(("gender_", "diet_preference_", "activity_level_", "bmi_category_", "allergies_", "deficiencies_", "diseases_")):
                row[col] = p_row[col]

        rows.append(row)

# Final dataset
training_df = pd.DataFrame(rows)
training_df.to_pickle("patient_food_training_data.pkl")


In [8]:
training_df.head()

,patient_id,food_id,age,weight,height,bmi,meal_type,food_calories,food_protein,food_fat,...,diet_preference_Veg,diet_preference_Vegan,activity_level_Active,activity_level_Light,activity_level_Moderate,activity_level_Sedentary,bmi_category_Normal,bmi_category_Obese,bmi_category_Overweight,bmi_category_Underweight
0,521,396,43,66,169,23.108435,snack,127.0,16.2,19.8,...,False,False,True,False,False,False,True,False,False,False
1,521,248,43,66,169,23.108435,lunch,140.0,3.0,6.0,...,False,False,True,False,False,False,True,False,False,False
2,521,215,43,66,169,23.108435,breakfast,140.0,3.5,1.0,...,False,False,True,False,False,False,True,False,False,False
3,521,353,43,66,169,23.108435,breakfast,348.9,26.4,28.2,...,False,False,True,False,False,False,True,False,False,False
4,521,548,43,66,169,23.108435,lunch,351.2,9.4,8.6,...,False,False,True,False,False,False,True,False,False,False


In [9]:
# import pandas as pd

# # Show 1500 rows without column truncation
# with pd.option_context(
#     'display.max_rows', 1500,        # Show up to 1500 rows
#     'display.max_columns', None,     # Show all columns
#     'display.width', None,           # Prevent line breaks
#     'display.max_colwidth', None     # Show full column content
# ):
#     print(training_df.head(1500))    # Display first 1500 rows


In [10]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Load training data
df = pd.read_pickle("patient_food_training_data.pkl")

# Drop identifiers and target column
X = df.drop(columns=["patient_id", "food_id", "label", "meal_type"])  # You can include meal_type if needed
y = df["label"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# XGBoost DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# Define parameters
params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "max_depth": 6,
    "eta": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "seed": 42
}

# Train
bst = xgb.train(params, dtrain, num_boost_round=100)

# Predict
y_pred_proba = bst.predict(dtest)
y_pred = (y_pred_proba > 0.5).astype(int)

# Evaluation
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Save model
bst.save_model("xgboost_patient_food_model.json")


Confusion Matrix:
[[58194   332]
 [ 7744  5730]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.99      0.94     58526
           1       0.95      0.43      0.59     13474

    accuracy                           0.89     72000
   macro avg       0.91      0.71      0.76     72000
weighted avg       0.89      0.89      0.87     72000



Model Training and Evaluation: Comparing ML Algorithms for Patient-Food Suitability Prediction

In [11]:
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
# import os

# from sklearn.model_selection import train_test_split
# from sklearn.metrics import (
#     classification_report, confusion_matrix, accuracy_score,
#     roc_auc_score, roc_curve, precision_recall_curve
# )
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import SVC
# from sklearn.neighbors import KNeighborsClassifier

# import xgboost as xgb
# import joblib

# # Create output directory
# os.makedirs("model_outputs", exist_ok=True)

# # Load data
# df = pd.read_pickle("patient_food_training_data.pkl")
# X = df.drop(columns=["patient_id", "food_id", "label", "meal_type"])
# y = df["label"]

# # Train-test split
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42, stratify=y
# )

# # Store results
# results = {}
# predictions = {}

# # 1. XGBoost
# dtrain = xgb.DMatrix(X_train, label=y_train)
# dtest = xgb.DMatrix(X_test, label=y_test)
# params = {
#     "objective": "binary:logistic",
#     "eval_metric": "logloss",
#     "max_depth": 6,
#     "eta": 0.1,
#     "subsample": 0.8,
#     "colsample_bytree": 0.8,
#     "seed": 42
# }
# bst = xgb.train(params, dtrain, num_boost_round=100)
# xgb_proba = bst.predict(dtest)
# xgb_pred = (xgb_proba > 0.5).astype(int)
# results["XGBoost"] = accuracy_score(y_test, xgb_pred)
# predictions["XGBoost"] = (xgb_pred, xgb_proba)

# # 2. Random Forest
# rf = RandomForestClassifier(n_estimators=100, random_state=42)
# rf.fit(X_train, y_train)
# rf_proba = rf.predict_proba(X_test)[:, 1]
# rf_pred = rf.predict(X_test)
# results["Random Forest"] = accuracy_score(y_test, rf_pred)
# predictions["Random Forest"] = (rf_pred, rf_proba)

# # 3. Logistic Regression
# lr = LogisticRegression(max_iter=1000)
# lr.fit(X_train, y_train)
# lr_proba = lr.predict_proba(X_test)[:, 1]
# lr_pred = lr.predict(X_test)
# results["Logistic Regression"] = accuracy_score(y_test, lr_pred)
# predictions["Logistic Regression"] = (lr_pred, lr_proba)

# # 4. SVM
# svm = SVC(probability=True)
# svm.fit(X_train, y_train)
# svm_proba = svm.predict_proba(X_test)[:, 1]
# svm_pred = svm.predict(X_test)
# results["SVM"] = accuracy_score(y_test, svm_pred)
# predictions["SVM"] = (svm_pred, svm_proba)

# # 5. KNN
# knn = KNeighborsClassifier(n_neighbors=5)
# knn.fit(X_train, y_train)
# knn_proba = knn.predict_proba(X_test)[:, 1]
# knn_pred = knn.predict(X_test)
# results["KNN"] = accuracy_score(y_test, knn_pred)
# predictions["KNN"] = (knn_pred, knn_proba)

# # ===============================
# # 🔍 Correlation Matrix Heatmap
# # ===============================
# plt.figure(figsize=(12, 10))
# sns.heatmap(X.corr(), annot=False, cmap='coolwarm', linewidths=0.5)
# plt.title("Feature Correlation Matrix")
# plt.tight_layout()
# plt.savefig("model_outputs/correlation_matrix.png")
# plt.close()

# # ===============================
# # 🔍 Classification Reports & Confusion Matrices
# # ===============================
# for name, (pred, _) in predictions.items():
#     # Classification report
#     report = classification_report(y_test, pred)
#     with open(f"model_outputs/{name.lower().replace(' ', '_')}_classification_report.txt", "w") as f:
#         f.write(f"{name} Classification Report\n\n")
#         f.write(report)
    
#     # Confusion matrix heatmap
#     cm = confusion_matrix(y_test, pred)
#     plt.figure(figsize=(4, 3))
#     sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
#     plt.title(f"{name} Confusion Matrix")
#     plt.xlabel("Predicted")
#     plt.ylabel("Actual")
#     plt.tight_layout()
#     plt.savefig(f"model_outputs/{name.lower().replace(' ', '_')}_confusion_matrix.png")
#     plt.close()

# # ===============================
# # 📊 Accuracy Bar Chart
# # ===============================
# plt.figure(figsize=(8, 5))
# sns.barplot(x=list(results.keys()), y=list(results.values()))
# plt.ylabel("Accuracy")
# plt.title("Model Accuracy Comparison")
# plt.xticks(rotation=45)
# plt.tight_layout()
# plt.savefig("model_outputs/accuracy_comparison.png")
# plt.close()

# # ===============================
# # 🎯 ROC Curves
# # ===============================
# plt.figure(figsize=(8, 6))
# for name, (_, proba) in predictions.items():
#     fpr, tpr, _ = roc_curve(y_test, proba)
#     auc = roc_auc_score(y_test, proba)
#     plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.2f})")
# plt.plot([0, 1], [0, 1], 'k--')
# plt.title("ROC Curve Comparison")
# plt.xlabel("False Positive Rate")
# plt.ylabel("True Positive Rate")
# plt.legend()
# plt.grid()
# plt.tight_layout()
# plt.savefig("model_outputs/roc_curves.png")
# plt.close()

# # ===============================
# # 📉 Precision-Recall Curves
# # ===============================
# plt.figure(figsize=(8, 6))
# for name, (_, proba) in predictions.items():
#     precision, recall, _ = precision_recall_curve(y_test, proba)
#     plt.plot(recall, precision, label=name)
# plt.title("Precision-Recall Curve Comparison")
# plt.xlabel("Recall")
# plt.ylabel("Precision")
# plt.legend()
# plt.grid()
# plt.tight_layout()
# plt.savefig("model_outputs/precision_recall_curves.png")
# plt.close()

# # ===============================
# # 💾 Save Best Model
# # ===============================
# best_model_name = max(results, key=results.get)
# print(f"\n✅ Best Model: {best_model_name} with Accuracy = {results[best_model_name]:.4f}")

# if best_model_name == "XGBoost":
#     bst.save_model("model_outputs/best_model_xgboost.json")
#     bst.save_model("xgboost_patient_food_model.json")
# elif best_model_name == "Random Forest":
#     joblib.dump(rf, "model_outputs/best_model_random_forest.pkl")
# elif best_model_name == "Logistic Regression":
#     joblib.dump(lr, "model_outputs/best_model_logistic.pkl")
# elif best_model_name == "SVM":
#     joblib.dump(svm, "model_outputs/best_model_svm.pkl")
# elif best_model_name == "KNN":
#     joblib.dump(knn, "model_outputs/best_model_knn.pkl")


Personalized 7-Day Meal Plan Generation Using Trained XGBoost Model

Unique Tags:
['Anemia', 'Calcium', 'Cholesterol', 'Diabetes', 'Fatty Liver', 'Folate (Vitamin B9)', 'Gastritis', 'Heart Disease', 'High Cholesterol', 'High Protein', 'Hypertension', 'Hypothyroidism', 'Immune Boost', 'Iron', 'Kidney Disease', 'Magnesium', 'Obesity', 'Omega 3', 'PCOS', 'PCOS/PCOD', 'Protein', 'Vitamin A', 'Vitamin B12', 'Vitamin D', 'Weight Loss', 'Zinc']

Unique Allergens:
['Egg', 'Eggs', 'Gluten', 'Lactose (Dairy)', 'Peanut', 'Peanuts', 'Seafood', 'Sesame', 'Shellfish', 'Soy', 'Tree Nut', 'Tree Nuts', 'peanut']


final script

In [1]:
import pandas as pd
import xgboost as xgb
import numpy as np
from collections import defaultdict
import re

model = xgb.Booster()

model.load_model("xgboost_patient_food_model.json")
foods = pd.read_pickle("processed_foods.pkl")
patients = pd.read_pickle("processed_patients.pkl")

def encode_patient_row(age, weight, height, gender, allergies, deficiencies, diseases, diet_pref, activity):
    bmi = weight / ((height / 100) ** 2)
    encoded = {
        "age": age,
        "weight": weight,
        "height": height,
        "bmi": bmi,
    }

    if bmi < 18.5:
        encoded["bmi_category_Underweight"] = 1
    elif bmi < 25:
        encoded["bmi_category_Normal"] = 1
    elif bmi < 30:
        encoded["bmi_category_Overweight"] = 1
    else:
        encoded["bmi_category_Obese"] = 1

    for col in patients.columns:
        if col.startswith("gender_"):
            encoded[col] = int(col == f"gender_{gender}")
        if col.startswith("diet_preference_"):
            encoded[col] = int(col == f"diet_preference_{diet_pref}")
        if col.startswith("activity_level_"):
            encoded[col] = int(col == f"activity_level_{activity}")
        if col.startswith("allergies_"):
            encoded[col] = int(col.replace("allergies_", "") in allergies)
        if col.startswith("deficiencies_"):
            encoded[col] = int(col.replace("deficiencies_", "") in deficiencies)
        if col.startswith("diseases_"):
            encoded[col] = int(col.replace("diseases_", "") in diseases)

    for col in patients.columns:
        if col not in encoded:
            encoded[col] = 0

    return pd.Series(encoded)

def estimate_maintenance_calories(age, weight, height, gender, activity_level):
    if gender.lower() == "male":
        bmr = 10 * weight + 6.25 * height - 5 * age + 5
    else:
        bmr = 10 * weight + 6.25 * height - 5 * age - 161

    activity_multipliers = {
        "sedentary": 1.2,
        "light": 1.375,
        "moderate": 1.55,
        "active": 1.725,
        "very active": 1.9
    }
    multiplier = activity_multipliers.get(activity_level.lower().strip(), 1.55)
    return bmr * multiplier

def matches_exceptions(text, exceptions):
    for exc in exceptions:
        if re.search(rf"\b{re.escape(exc)}\b", text, flags=re.IGNORECASE):
            return True
    return False

def generate_meal_plan_from_inputs(
    patient_name,
    age, weight, height, gender,
    allergies_str, deficiencies_str, diseases_str,
    diet_pref, activity,
    exceptions_str="",
    meals_per_day=["breakfast", "lunch", "snack", "dinner"],
    threshold=0.3,
    top_k=10,
):
    allergies = allergies_str.split(";") if allergies_str else []
    deficiencies = deficiencies_str.split(";") if deficiencies_str else []
    diseases = diseases_str.split(";") if diseases_str else []
    exceptions = [e.strip().lower() for e in exceptions_str.split(";") if e.strip()]

    patient = encode_patient_row(age, weight, height, gender, allergies, deficiencies, diseases, diet_pref, activity)
    maintenance_calories = estimate_maintenance_calories(age, weight, height, gender, activity)
    meals_per_day = [m.lower().strip() for m in meals_per_day]

    food_rows = []
    for _, food in foods.iterrows():
        if food["meal_type"].lower().strip() not in meals_per_day:
            continue

        food_name_lower = str(food["name"]).lower()
        ingredients = str(food.get("ingredients", "")).lower()

        # Handle diet_Type field which might be string or list
        food_diet = ""
        dtype_val = food.get("diet_Type")
        if isinstance(dtype_val, list) and len(dtype_val) > 0:
            food_diet = dtype_val[0].lower()
        elif isinstance(dtype_val, str):
            food_diet = dtype_val.lower()

        # Filter exceptions
        if matches_exceptions(food_name_lower, exceptions) or matches_exceptions(ingredients, exceptions) or matches_exceptions(food_diet, exceptions):
            continue

        # *** ADD DIET FILTER HERE ***
        if diet_pref.lower() == "veg" and food_diet == "non-veg":
            continue  # skip non-veg foods for vegetarians
        if diet_pref.lower() == "vegan" and food_diet in ["non-veg", "veg"]:
            continue  # skip non-veg and veg for vegans

        row = {
            "age": patient["age"],
            "weight": patient["weight"],
            "height": patient["height"],
            "bmi": patient["bmi"],
            "food_calories": food["calories"],
            "food_protein": food["protein"],
            "food_fat": food["fat"],
            "food_carbs": food["carbs"]
        }

        for col in patients.columns:
            if col.startswith(("gender_", "diet_preference_", "activity_level_", "bmi_category_", "allergies_", "deficiencies_", "diseases_")):
                row[col] = patient[col]

        food_rows.append((food["name"], food["meal_type"].lower().strip(), row))

    if not food_rows:
        return {"error": "No foods match the criteria or were excluded by exceptions."}

    features_df = pd.DataFrame([r[2] for r in food_rows])
    dmatrix = xgb.DMatrix(features_df)
    preds = model.predict(dmatrix)

    meal_plan = defaultdict(list)
    for (food_name, meal_type, _), score in zip(food_rows, preds):
        if score >= threshold:
            meal_plan[meal_type].append((food_name, score))

    final_plan = {meal: [] for meal in meals_per_day}
    priority_map = {"non-veg": 0, "veg": 1, "vegan": 2}
    max_candidates_per_meal = 15
    meal_candidates = {}

    for meal in meals_per_day:
        candidates = meal_plan.get(meal, [])
        candidates = sorted(candidates, key=lambda x: x[1], reverse=True)

        def diet_priority(food_name):
            dtype_series = foods.loc[foods["name"] == food_name, "diet_Type"]
            if dtype_series.empty:
                return 3
            dtype_val = dtype_series.values[0]
            if isinstance(dtype_val, list) and len(dtype_val) > 0:
                dtype = dtype_val[0].lower()
            elif isinstance(dtype_val, str):
                dtype = dtype_val.lower()
            else:
                return 3
            return priority_map.get(dtype, 3)

        candidates = sorted(candidates, key=lambda x: diet_priority(x[0]))
        if not candidates:
            candidates = sorted(
                [(food_name, score) for (food_name, m_type, _), score in zip(food_rows, preds) if m_type == meal],
                key=lambda x: x[1], reverse=True
            )

        meal_candidates[meal] = candidates[:max_candidates_per_meal] if candidates else []

    calorie_split = {
        "breakfast": 0.35,
        "lunch": 0.30,
        "snack": 0.10,
        "dinner": 0.25
    }

    used_meals = {meal: set() for meal in meals_per_day}
    structured_plan = {
        "patient_name": patient_name,
        "bmi": patient["bmi"],
        "bmi_category": (
            "Underweight" if patient["bmi"] < 18.5 else
            "Normal" if patient["bmi"] < 25 else
            "Overweight" if patient["bmi"] < 30 else "Obese"
        ),
        "maintenance_calories": int(maintenance_calories),
        "days": []
    }

    for day_idx in range(7):
        day_plan = {}
        day_calories = 0
        for meal in meals_per_day:
            candidates = meal_candidates.get(meal, [])
            if not candidates:
                day_plan[meal] = {"food": "No suitable option", "serving": "N/A", "calories": 0}
                continue

            required_cals = maintenance_calories * calorie_split.get(meal, 0.25)
            best_fit = None

            filtered_candidates = [c for c in candidates if c[0] not in used_meals[meal]]
            for food_name, score in filtered_candidates or candidates:
                cal_per_100g = foods.loc[foods["name"] == food_name, "calories"].values[0]
                serving_size = (required_cals / cal_per_100g) * 100
                if serving_size <= 500:
                    total_cals = (cal_per_100g / 100) * serving_size
                    best_fit = (food_name, int(np.ceil(serving_size)), int(np.ceil(total_cals)))
                    break

            if best_fit:
                food_name, serving, calories = best_fit
            else:
                food_name = candidates[0][0]
                cal_per_100g = foods.loc[foods["name"] == food_name, "calories"].values[0]
                serving = int(np.ceil((required_cals / cal_per_100g) * 100))
                calories = int(np.ceil((cal_per_100g / 100) * serving))

            used_meals[meal].add(food_name)
            day_plan[meal] = {
                "food": food_name,
                "serving": f"{serving}g",
                "calories": calories
            }
            day_calories += calories

        day_plan["total_calories"] = day_calories
        structured_plan["days"].append(day_plan)

    return structured_plan

# Example usage
plan = generate_meal_plan_from_inputs(
    patient_name="Subhankar Pani",
    age=25,
    weight=80,
    height=167,
    gender="male",
    allergies_str="",
    deficiencies_str="",
    diseases_str="",
    diet_pref="veg",
    activity="Moderate",
    exceptions_str="pork"
)

# Print formatted result
if "error" in plan:
    print("Error:", plan["error"])
else:
    print(f"Patient: {plan['patient_name']}")
    print(f"BMI: {plan['bmi']:.2f} ({plan['bmi_category']})")
    print(f"Maintenance calories per day: {plan['maintenance_calories']}")
    print("=" * 30)
    for i, day in enumerate(plan["days"], 1):
        print(f"Day {i}:")
        for meal, details in day.items():
            if meal == "total_calories":
                print(f"  Total Calories: {details} kcal")
            else:
                print(f"  {meal.title()}: {details['food']} ({details['serving']} / {details['calories']} kcal)")
        print("-" * 30)
    print("=" * 30)


Patient: Subhankar Pani
BMI: 28.69 (Overweight)
Maintenance calories per day: 2671
Day 1:
  Breakfast: Spinach Soup (167g / 936 kcal)
  Lunch: Sattu Drink (175g / 802 kcal)
  Snack: Rajma Chawal (48g / 268 kcal)
  Dinner: Karela Sabzi (127g / 668 kcal)
  Total Calories: 2674 kcal
------------------------------
Day 2:
  Breakfast: Paneer Tikka (197g / 936 kcal)
  Lunch: Cauliflower Currywith Roti (363g / 802 kcal)
  Snack: Pav baji (69g / 268 kcal)
  Dinner: Kadhi Bari (304g / 668 kcal)
  Total Calories: 2674 kcal
------------------------------
Day 3:
  Breakfast: Paratha (499g / 936 kcal)
  Lunch: Rajma Chawal (142g / 802 kcal)
  Snack: Vegetable Upma (58g / 268 kcal)
  Dinner: Mixed Veg Curry (478g / 668 kcal)
  Total Calories: 2674 kcal
------------------------------
Day 4:
  Breakfast: Chana Salad (229g / 936 kcal)
  Lunch: Green Smoothie (193g / 802 kcal)
  Snack: Amla Juice (62g / 268 kcal)
  Dinner: Date Energy Balls (286g / 668 kcal)
  Total Calories: 2674 kcal
-----------------